# 10 - Improved Peak-Hour Random Forest

This notebook improves the direct peak-hour model without touching the test set.
It adds building identity and multi-day historical patterns, tunes the Random Forest
on a chronological holdout taken only from the training period, and evaluates the
selected configuration once on validation.

**Primary selection metric:** accuracy within one hour of the actual daily peak.

## 1. Setup

In [ ]:
from pathlib import Path
from time import perf_counter

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import f1_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

pd.set_option("display.max_columns", None)

RANDOM_STATE = 42
project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
modeling_dir = project_root / "data" / "processed" / "modeling"
reports_dir = project_root / "reports"
figures_dir = project_root / "figures"
reports_dir.mkdir(parents=True, exist_ok=True)
figures_dir.mkdir(parents=True, exist_ok=True)

## 2. Build one row per building-day

In [ ]:
hourly_columns = [
    "building_id", "timestamp", "meter_reading", "site_id", "primary_use",
    "square_feet", "year_built", "air_temperature", "cloud_coverage",
    "dew_temperature", "precip_depth_1_hr", "sea_level_pressure",
    "wind_speed", "hour", "day_of_week", "month", "is_weekend",
]


def build_daily_table(path: Path, split_name: str) -> pd.DataFrame:
    hourly = pd.read_parquet(path, columns=hourly_columns)
    hourly["date"] = hourly["timestamp"].dt.normalize()
    hourly = hourly.sort_values(["building_id", "timestamp"]).reset_index(drop=True)
    grouped = hourly.groupby(["building_id", "date"], observed=True)

    daily = grouped.agg(
        hourly_records=("hour", "count"),
        unique_hours=("hour", "nunique"),
        unique_readings=("meter_reading", "nunique"),
        daily_mean=("meter_reading", "mean"),
        daily_max=("meter_reading", "max"),
        daily_min=("meter_reading", "min"),
        daily_std=("meter_reading", "std"),
        site_id=("site_id", "first"),
        primary_use=("primary_use", "first"),
        square_feet=("square_feet", "first"),
        year_built=("year_built", "first"),
        day_of_week=("day_of_week", "first"),
        month=("month", "first"),
        is_weekend=("is_weekend", "first"),
        temperature_mean=("air_temperature", "mean"),
        temperature_max=("air_temperature", "max"),
        temperature_min=("air_temperature", "min"),
        dew_temperature_mean=("dew_temperature", "mean"),
        cloud_coverage_mean=("cloud_coverage", "mean"),
        precipitation_total=("precip_depth_1_hr", "sum"),
        sea_level_pressure_mean=("sea_level_pressure", "mean"),
        wind_speed_mean=("wind_speed", "mean"),
        wind_speed_max=("wind_speed", "max"),
    ).reset_index()

    peak_indices = grouped["meter_reading"].idxmax()
    peaks = hourly.loc[
        peak_indices.to_numpy(), ["building_id", "date", "hour"]
    ].rename(columns={"hour": "peak_hour"})

    profile = hourly.pivot_table(
        index=["building_id", "date"],
        columns="hour",
        values="meter_reading",
        aggfunc="first",
    ).reindex(columns=range(24))
    profile.columns = [f"reading_h{hour:02d}" for hour in range(24)]
    profile = profile.reset_index()

    daily = daily.merge(peaks, on=["building_id", "date"], validate="one_to_one")
    daily = daily.merge(profile, on=["building_id", "date"], validate="one_to_one")
    daily["split"] = split_name
    return daily


train_daily = build_daily_table(modeling_dir / "model_train.parquet", "train")
validation_daily = build_daily_table(
    modeling_dir / "model_validation.parquet", "validation"
)
daily_data = (
    pd.concat([train_daily, validation_daily], ignore_index=True)
    .sort_values(["building_id", "date"])
    .reset_index(drop=True)
)

pd.DataFrame({
    "split": ["train", "validation"],
    "building_days": [len(train_daily), len(validation_daily)],
})

## 3. Add building-aware historical features

In [ ]:
grouped = daily_data.groupby("building_id", observed=True)
reading_columns = [f"reading_h{hour:02d}" for hour in range(24)]

# Preserve the previous day's complete hourly shape used by the baseline model.
previous_columns = [
    "daily_mean", "daily_max", "daily_min", "daily_std", "peak_hour",
    "hourly_records",
] + reading_columns
previous_date = grouped["date"].shift(1)
previous_is_consecutive = (daily_data["date"] - previous_date) == pd.Timedelta(days=1)
for column in previous_columns:
    name = f"previous_{column}"
    daily_data[name] = grouped[column].shift(1)
    daily_data.loc[~previous_is_consecutive, name] = np.nan

daily_data["previous_day_complete"] = (
    daily_data["previous_hourly_records"] == 24
).astype("int8")

# Add recent peak and consumption history without using the target day itself.
for lag in [1, 2, 3, 7]:
    lag_date = grouped["date"].shift(lag)
    valid_lag = (daily_data["date"] - lag_date) == pd.Timedelta(days=lag)
    lag_peak = grouped["peak_hour"].shift(lag).where(valid_lag)
    daily_data[f"peak_lag_{lag}_sin"] = np.sin(2 * np.pi * lag_peak / 24)
    daily_data[f"peak_lag_{lag}_cos"] = np.cos(2 * np.pi * lag_peak / 24)

    for column in ["daily_mean", "daily_max", "daily_std"]:
        daily_data[f"{column}_lag_{lag}"] = grouped[column].shift(lag).where(valid_lag)

peak_sin = np.sin(2 * np.pi * daily_data["peak_hour"] / 24)
peak_cos = np.cos(2 * np.pi * daily_data["peak_hour"] / 24)
daily_data["recent_peak_sin"] = peak_sin.groupby(
    daily_data["building_id"], observed=True
).transform(lambda values: values.shift(1).rolling(7, min_periods=2).mean())
daily_data["recent_peak_cos"] = peak_cos.groupby(
    daily_data["building_id"], observed=True
).transform(lambda values: values.shift(1).rolling(7, min_periods=2).mean())

daily_data["log_square_feet"] = np.log1p(daily_data["square_feet"])
daily_data["building_age"] = (2016 - daily_data["year_built"]).clip(lower=0)
daily_data["eligible_target_day"] = (
    (daily_data["hourly_records"] == 24)
    & (daily_data["unique_hours"] == 24)
    & (daily_data["unique_readings"] > 1)
)

daily_data.groupby("split", observed=True)["eligible_target_day"].agg(
    all_building_days="count",
    eligible_building_days="sum",
    eligible_fraction="mean",
)

## 4. Prepare chronological tuning and validation sets

In [ ]:
previous_profile_features = [f"previous_reading_h{hour:02d}" for hour in range(24)]
history_features = []
for lag in [1, 2, 3, 7]:
    history_features.extend([
        f"peak_lag_{lag}_sin", f"peak_lag_{lag}_cos",
        f"daily_mean_lag_{lag}", f"daily_max_lag_{lag}", f"daily_std_lag_{lag}",
    ])

numeric_features = [
    "log_square_feet", "building_age", "day_of_week", "month", "is_weekend",
    "temperature_mean", "temperature_max", "temperature_min",
    "dew_temperature_mean", "cloud_coverage_mean", "precipitation_total",
    "sea_level_pressure_mean", "wind_speed_mean", "wind_speed_max",
    "previous_daily_mean", "previous_daily_max", "previous_daily_min",
    "previous_daily_std", "previous_day_complete", "recent_peak_sin",
    "recent_peak_cos",
] + history_features + previous_profile_features
categorical_features = ["building_id", "site_id", "primary_use"]
feature_columns = numeric_features + categorical_features

eligible = daily_data[daily_data["eligible_target_day"]].copy()
train_data = eligible[eligible["split"] == "train"].copy()
validation_data = eligible[eligible["split"] == "validation"].copy()

unique_train_dates = np.sort(train_data["date"].unique())
tuning_cutoff = pd.Timestamp(unique_train_dates[int(len(unique_train_dates) * 0.80)])
inner_train = train_data[train_data["date"] < tuning_cutoff].copy()
tuning_holdout = train_data[train_data["date"] >= tuning_cutoff].copy()


def make_preprocessor() -> ColumnTransformer:
    return ColumnTransformer(
        transformers=[
            (
                "numeric",
                Pipeline([
                    ("imputer", SimpleImputer(strategy="median")),
                    ("scaler", StandardScaler()),
                ]),
                numeric_features,
            ),
            (
                "categorical",
                Pipeline([
                    ("imputer", SimpleImputer(strategy="most_frequent")),
                    ("onehot", OneHotEncoder(
                        handle_unknown="ignore", sparse_output=True, dtype=np.float32
                    )),
                ]),
                categorical_features,
            ),
        ],
        sparse_threshold=1.0,
    )


tuning_preprocessor = make_preprocessor()
X_inner = tuning_preprocessor.fit_transform(inner_train[feature_columns]).astype("float32")
X_tuning = tuning_preprocessor.transform(tuning_holdout[feature_columns]).astype("float32")
y_inner = inner_train["peak_hour"].to_numpy(dtype="int8")
y_tuning = tuning_holdout["peak_hour"].to_numpy(dtype="int8")

pd.Series({
    "inner_training_days": len(inner_train),
    "tuning_holdout_days": len(tuning_holdout),
    "validation_days_reserved": len(validation_data),
    "tuning_cutoff": tuning_cutoff.date(),
    "encoded_features": X_inner.shape[1],
})

## 5. Tune the Random Forest on training data only

In [ ]:
def circular_errors(actual: np.ndarray, predicted: np.ndarray) -> np.ndarray:
    difference = np.abs(actual.astype(int) - predicted.astype(int))
    return np.minimum(difference, 24 - difference)


def choose_peak(probabilities: np.ndarray, classes: np.ndarray, rule: str) -> np.ndarray:
    full_probabilities = np.zeros((len(probabilities), 24), dtype=np.float32)
    full_probabilities[:, classes.astype(int)] = probabilities
    if rule == "neighbor_smoothed":
        scores = (
            np.roll(full_probabilities, 1, axis=1)
            + full_probabilities
            + np.roll(full_probabilities, -1, axis=1)
        )
    else:
        scores = full_probabilities
    return scores.argmax(axis=1).astype("int8")


candidates = [
    {
        "configuration": "baseline_parameters",
        "n_estimators": 250,
        "max_depth": None,
        "min_samples_leaf": 2,
        "max_features": "sqrt",
        "class_weight": None,
    },
    {
        "configuration": "deeper_trees",
        "n_estimators": 300,
        "max_depth": None,
        "min_samples_leaf": 1,
        "max_features": "sqrt",
        "class_weight": None,
    },
    {
        "configuration": "broader_feature_search",
        "n_estimators": 300,
        "max_depth": 28,
        "min_samples_leaf": 2,
        "max_features": 0.15,
        "class_weight": None,
    },
    {
        "configuration": "balanced_classes",
        "n_estimators": 300,
        "max_depth": None,
        "min_samples_leaf": 2,
        "max_features": "sqrt",
        "class_weight": "balanced_subsample",
    },
]

tuning_rows = []
for parameters in candidates:
    model_parameters = {key: value for key, value in parameters.items() if key != "configuration"}
    print(f"Training {parameters['configuration']}...")
    model = RandomForestClassifier(
        **model_parameters,
        n_jobs=-1,
        random_state=RANDOM_STATE,
    )
    start = perf_counter()
    model.fit(X_inner, y_inner)
    fit_seconds = perf_counter() - start
    probabilities = model.predict_proba(X_tuning)

    for rule in ["raw_argmax", "neighbor_smoothed"]:
        predictions = choose_peak(probabilities, model.classes_, rule)
        errors = circular_errors(y_tuning, predictions)
        tuning_rows.append({
            **parameters,
            "decision_rule": rule,
            "exact_peak_accuracy": (errors == 0).mean(),
            "within_1_hour_accuracy": (errors <= 1).mean(),
            "within_2_hours_accuracy": (errors <= 2).mean(),
            "mean_peak_hour_error": errors.mean(),
            "macro_f1": f1_score(y_tuning, predictions, average="macro"),
            "fit_seconds": fit_seconds,
        })

tuning_results = pd.DataFrame(tuning_rows).sort_values(
    ["within_1_hour_accuracy", "exact_peak_accuracy"], ascending=False
).reset_index(drop=True)
tuning_results.to_csv(
    reports_dir / "peak_hour_tuning_search_results.csv", index=False
)
tuning_results

## 6. Refit the selected model and evaluate validation once

In [ ]:
selected = tuning_results.iloc[0]
selected_parameters = {
    "n_estimators": int(selected["n_estimators"]),
    "max_depth": None if pd.isna(selected["max_depth"]) else int(selected["max_depth"]),
    "min_samples_leaf": int(selected["min_samples_leaf"]),
    "max_features": selected["max_features"],
    "class_weight": None if pd.isna(selected["class_weight"]) else selected["class_weight"],
}

final_preprocessor = make_preprocessor()
X_train = final_preprocessor.fit_transform(train_data[feature_columns]).astype("float32")
X_validation = final_preprocessor.transform(
    validation_data[feature_columns]
).astype("float32")
y_train = train_data["peak_hour"].to_numpy(dtype="int8")
y_validation = validation_data["peak_hour"].to_numpy(dtype="int8")

final_model = RandomForestClassifier(
    **selected_parameters,
    n_jobs=-1,
    random_state=RANDOM_STATE,
)
start = perf_counter()
final_model.fit(X_train, y_train)
fit_seconds = perf_counter() - start
validation_probabilities = final_model.predict_proba(X_validation)
validation_predictions = choose_peak(
    validation_probabilities,
    final_model.classes_,
    selected["decision_rule"],
)
validation_errors = circular_errors(y_validation, validation_predictions)

improved_metrics = pd.DataFrame([{
    "model": "Improved Random Forest",
    "configuration": selected["configuration"],
    "decision_rule": selected["decision_rule"],
    "evaluated_building_days": len(validation_data),
    "exact_peak_accuracy": (validation_errors == 0).mean(),
    "within_1_hour_accuracy": (validation_errors <= 1).mean(),
    "within_2_hours_accuracy": (validation_errors <= 2).mean(),
    "mean_peak_hour_error": validation_errors.mean(),
    "median_peak_hour_error": np.median(validation_errors),
    "macro_f1": f1_score(y_validation, validation_predictions, average="macro"),
    "fit_seconds": fit_seconds,
}])
improved_metrics.to_csv(
    reports_dir / "improved_peak_hour_validation_metrics.csv", index=False
)
improved_metrics

## 7. Compare with the original model

In [ ]:
original_metrics = pd.read_csv(reports_dir / "peak_hour_validation_metrics.csv")
original_rf = original_metrics.loc[original_metrics["model"] == "Random Forest"].iloc[0]

comparison = pd.DataFrame([
    {
        "model": "Original Random Forest",
        "exact_peak_accuracy": original_rf["exact_peak_accuracy"],
        "within_1_hour_accuracy": original_rf["within_1_hour_accuracy"],
        "within_2_hours_accuracy": original_rf["within_2_hours_accuracy"],
        "mean_peak_hour_error": original_rf["mean_peak_hour_error"],
    },
    improved_metrics.iloc[0][[
        "model", "exact_peak_accuracy", "within_1_hour_accuracy",
        "within_2_hours_accuracy", "mean_peak_hour_error",
    ]].to_dict(),
])
comparison["within_1_improvement_points"] = (
    comparison["within_1_hour_accuracy"]
    - float(original_rf["within_1_hour_accuracy"])
) * 100
comparison.to_csv(
    reports_dir / "original_vs_improved_peak_hour_metrics.csv", index=False
)
comparison

In [ ]:
plot_data = comparison.copy()
positions = np.arange(len(plot_data))
width = 0.34

fig, ax = plt.subplots(figsize=(9, 5.5), dpi=120)
exact_bars = ax.bar(
    positions - width / 2,
    plot_data["exact_peak_accuracy"] * 100,
    width,
    color="#b8b8b8",
    label="Exact hour",
)
within_bars = ax.bar(
    positions + width / 2,
    plot_data["within_1_hour_accuracy"] * 100,
    width,
    color="#1f4e79",
    label="Within one hour",
)
ax.set_title("Building-Aware Peak-Hour Model Compared With Baseline", fontsize=16, weight="bold")
ax.set_ylabel("Accuracy (%)")
ax.set_xticks(positions)
ax.set_xticklabels(plot_data["model"])
ax.legend()
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.grid(False)

for bars in [exact_bars, within_bars]:
    for bar in bars:
        value = bar.get_height()
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            value + 0.4,
            f"{value:.1f}%",
            ha="center",
            fontsize=10,
        )

plt.tight_layout()
fig.savefig(
    figures_dir / "original_vs_improved_peak_hour_model.png",
    dpi=150,
    bbox_inches="tight",
)
plt.show()

## Interpretation

The tuning stage uses only the training period. Validation remains untouched until
one configuration and one probability decision rule have been selected. The project
test split is still reserved for the final evaluation.

Daily weather summaries assume that same-day weather observations or forecasts are
available when producing the peak-hour prediction.